In [4]:
import re
import html
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")



In [3]:
df_clean = pd.read_csv("/Users/EXOTISCH TECH/Desktop/sms-spam-detector/data/processed/sms_dedup.csv")
df_clean.head()

,label,message,label_num,char_count,word_count
0,ham,"Go until jurong point, crazy.. Available only ...",0,111,20
1,ham,Ok lar... Joking wif u oni...,0,29,6
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1,155,28
3,ham,U dun say so early hor... U c already then say...,0,49,11
4,ham,"Nah I don't think he goes to usf, he lives aro...",0,61,13


## Text Preprocessing

In [5]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def clean_text(text: str) -> str:
    text = html.unescape(str(text))                              # &amp; -> &, &lt; -> <
    text = re.sub(r"<[^>]+>", " ", text)                         # tokens like <#>, <URL>
    text = re.sub(r"(http\S+|www\.\S+)", " urltoken ", text)     # URLs
    text = re.sub(r"\b\d{9,}\b", " phonetoken ", text)           # long numbers (phones/short codes)
    text = re.sub(r"[£$€]\s?\d[\d,\.]*", " currencytoken ", text)  # money amounts
    text = re.sub(r"\d+", " numtoken ", text)                    # remaining numbers
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)                        # drop punctuation/symbols
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

# Quick sanity check
samples = df_clean["message"].sample(5, random_state=1)
for s in samples:
    print("RAW  :", s)
    print("CLEAN:", clean_text(s), "\n")

RAW  : Sorry da..today i wont come to play..i have driving clas..
CLEAN: sorri da today wont come play drive cla 

RAW  : How have your little darlings been so far this week? Need a coffee run tomo?Can't believe it's that time of week already …
CLEAN: littl darl far week need coffe run tomo believ time week alreadi 

RAW  : Yunny i'm walking in citylink now ü faster come down... Me very hungry...
CLEAN: yunni walk citylink faster come hungri 

RAW  : Free video camera phones with Half Price line rental for 12 mths and 500 cross ntwk mins 100 txts. Call MobileUpd8 08001950382 or Call2OptOut/674&
CLEAN: free video camera phone half price line rental numtoken mth numtoken cross ntwk min numtoken txt call mobileupd numtoken phonetoken call numtoken optout numtoken 

RAW  : Orh i tot u say she now still dun believe.
CLEAN: orh tot say still dun believ 



In [6]:
df_clean["clean_message"] = df_clean["message"].apply(clean_text)

# Some messages may become empty after cleaning
empty = (df_clean["clean_message"].str.strip() == "").sum()
print("Empty after cleaning:", empty)
df_clean = df_clean[df_clean["clean_message"].str.strip() != ""].reset_index(drop=True)

df_clean[["label", "message", "clean_message"]].head()

Empty after cleaning: 9


,label,message,clean_message
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",go jurong point crazi avail bugi great world la buffet cine got amor wat
1,ham,Ok lar... Joking wif u oni...,ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std t...,free entri numtoken wkli comp win fa cup final tkt numtoken st may numtoken text fa numtoken receiv entri question s...
3,ham,U dun say so early hor... U c already then say...,dun say earli hor alreadi say
4,ham,"Nah I don't think he goes to usf, he lives around here though",nah think goe usf live around though


## Train / Test split

In [11]:
# splitting the data
X = df_clean["clean_message"]
y = df_clean["label_num"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])
print("Spam % train:", round(y_train.mean() * 100, 2))
print("Spam % test :", round(y_test.mean() * 100, 2))

Train: 4120 | Test: 1031
Spam % train: 12.48
Spam % test : 12.42


In [12]:
train_df = df_clean.loc[X_train.index, ["message", "clean_message", "label_num"]]
test_df  = df_clean.loc[X_test.index,  ["message", "clean_message", "label_num"]]
train_df.to_csv("/Users/EXOTISCH TECH/Desktop/sms-spam-detector/data/processed/train.csv", index=False)
test_df.to_csv("/Users/EXOTISCH TECH/Desktop/sms-spam-detector/data/processed/test.csv", index=False)